# Station Stacking v8 - KLAX

Experimental notebook for `KLAX`.

This version keeps the v7 live-safe GFS/HRRR/NBM contract, adds source-owned remaining-warmup feature engineering, and drops only conservative zero-importance input fields. Artifacts are written to `data/calibration/station_stacking_v8`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KLAX"
PROVIDERS = ("gfs", "hrrr", "nbm")
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 50
STACK_OPTUNA_TRIALS = 50
OPTUNA_STARTUP_TRIALS = 20
STACK_OPTUNA_STARTUP_TRIALS = 20
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.calibration.station_stacking import (
    StationStackingConfig,
    V8_DROPPED_FEATURE_COLUMNS,
    V8_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V8 Contract

`feature_version="v8"` keeps the v7 live-safe NBM setup and direct `actual_high_f` target. V8 adds remaining-warmup features and removes only conservative zero-importance model inputs from the feature matrix.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V8_FEATURE_COLUMNS, sorted(V8_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
12,KLAX,gfs,1982,2021-01-01,2026-06-10
13,KLAX,hrrr,1987,2021-01-01,2026-06-10
14,KLAX,nbm,1986,2021-01-01,2026-06-10


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v8",
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v8/KLAX_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-15 05:47:53,502] Using an existing study with name 'KLAX_v8_base_xgboost_mae_f' instead of creating a new one.
[I 2026-06-15 05:48:07,413] Trial 2 finished with value: 1.621816818347487 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.621816818347487.
[I 2026-06-15 05:51:48,343] Trial 3 finished with value: 1.5086254581820675 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 3 with value: 1.5086254581820675.
[I 2026-06-15 05:53:13,

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,730,1.652945,2.916879
1,validation_2024_2025,lightgbm,730,1.618821,2.917260
2,validation_2024_2025,catboost,730,1.495312,2.804837
3,validation_2024_2025,hrrr_raw,730,1.824908,3.106160
4,validation_2024_2025,gfs_raw,730,3.102706,4.345859
5,test_2026,xgboost,137,2.462660,3.625580
6,test_2026,lightgbm,137,1.474627,1.835297
7,test_2026,catboost,137,2.364067,2.968886
8,test_2026,ridge_stack,137,1.821198,2.252137
9,test_2026,hrrr_raw,137,1.534561,2.020557


## V8 Feature Coverage


In [8]:
v8_feature_coverage = (
    result.features[V8_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v8_feature_coverage


,feature,coverage_pct
0,v2_morning_warmup_to_consensus_f,100.000000
1,v3_high_so_far_above_current_f,100.000000
2,v2_humidity_warmup_interaction,100.000000
3,v2_spread_per_warmup_f,100.000000
4,v3_humidity_remaining_warmup_interaction,100.000000
5,v3_remaining_warmup_per_spread_f,100.000000
6,v4_any_forecast_precip,100.000000
7,v3_remaining_warmup_from_high_so_far_f,100.000000
8,observed_high_so_far_change_since_9am_f,100.000000
9,observed_morning_warmup_rate_f_per_hour,100.000000


In [9]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V8_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
163,v2_recent_heat_anomaly_f,numeric
164,v2_recent_heat_momentum_f,numeric
165,v2_morning_warmup_to_consensus_f,numeric
166,v2_consensus_minus_7d_actual_f,numeric
167,v2_spread_per_warmup_f,numeric
168,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [10]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V8_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


NameError: name 'TREND_COLUMNS' is not defined

## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
2,oof_2026,hrrr_raw,137,83,60.583942
3,oof_2026,lightgbm,137,78,56.934307
5,oof_2026,ridge_stack,137,66,48.175182
6,oof_2026,xgboost,137,66,48.175182
4,oof_2026,nbm_raw,137,60,43.795620
0,oof_2026,catboost,137,55,40.145985
1,oof_2026,gfs_raw,137,38,27.737226
7,validation_2024_2025,catboost,730,468,64.109589
10,validation_2024_2025,lightgbm,730,434,59.452055
12,validation_2024_2025,xgboost,730,417,57.123288


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,xgboost,78,1.176843,1.523835,v6
1,test_2026,ridge_stack,78,1.265691,1.581265,v6
2,test_2026,lightgbm,78,1.280411,1.654181,v6
3,test_2026,catboost,78,1.361088,1.835915,v6
4,test_2026,catboost,78,1.420672,1.891794,v5
...,...,...,...,...,...,...
61,validation_2024_2025,gfs_raw,592,3.404574,5.253908,v7
62,validation_2024_2025,gfs_raw,533,3.660056,5.474728,v5
63,validation_2024_2025,gfs_raw,533,3.660056,5.474728,v6
64,validation_2024_2025,gfs_raw,664,3.666696,5.648337,v1


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,137,2.462660,3.625580,30.656934
1,lightgbm,137,1.474627,1.835297,37.956204
2,catboost,137,2.364067,2.968886,27.007299
3,ridge_stack,137,1.821198,2.252137,30.656934
4,hrrr_raw,137,1.534561,2.020557,43.79562
5,gfs_raw,137,3.172394,3.790030,16.788321
